In [ ]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import webbrowser
import json
import html

# ============================================================
# TASK 5: GROUPED BAR CHART
# Average Rating vs Total Review Count by App Category
# Dashboard visible only from 3 PM to 5 PM IST
# ============================================================

# 1. Load dataset
DATA_FILE = Path("Play Store Data.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Play Store Data.csv was not found. "
        "Keep it in the same project folder as this notebook."
    )

df = pd.read_csv(DATA_FILE)
print("Dataset loaded:", df.shape)

# 2. Clean numeric columns
def clean_number(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

def size_to_mb(value):
    if pd.isna(value):
        return float("nan")

    value = str(value).strip().upper()

    if value in ("", "NAN", "VARIES WITH DEVICE"):
        return float("nan")

    try:
        if value.endswith("M"):
            return float(value[:-1])
        elif value.endswith("K"):
            return float(value[:-1]) / 1024
        elif value.endswith("G"):
            return float(value[:-1]) * 1024
        else:
            return float(value)
    except ValueError:
        return float("nan")

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df["Reviews"] = clean_number(df["Reviews"])
df["Installs"] = clean_number(df["Installs"])
df["Size_MB"] = df["Size"].apply(size_to_mb)

df["Last Updated"] = pd.to_datetime(
    df["Last Updated"], errors="coerce"
)

# 3. Apply app-level filters:
#    Size at least 10 MB and last updated in January
filtered = df[
    (df["Size_MB"] >= 10) &
    (df["Last Updated"].dt.month == 1) &
    df["Category"].notna() &
    df["Rating"].notna() &
    df["Reviews"].notna() &
    df["Installs"].notna()
].copy()

print("Apps after size and January filters:", len(filtered))

# 4. Aggregate metrics by category
category_stats = (
    filtered.groupby("Category")
    .agg(
        Average_Rating=("Rating", "mean"),
        Total_Reviews=("Reviews", "sum"),
        Total_Installs=("Installs", "sum")
    )
    .reset_index()
)

# Exclude categories with average rating below 4.0
category_stats = category_stats[
    category_stats["Average_Rating"] >= 4.0
].copy()

# 5. Select top 10 eligible categories by total installs
top10 = (
    category_stats.sort_values("Total_Installs", ascending=False)
    .head(10)
    .sort_values("Total_Installs", ascending=False)
    .reset_index(drop=True)
)

if top10.empty:
    raise ValueError(
        "No categories matched all filters. "
        "Check the January, size, and rating conditions."
    )

print("\nTop categories for Task 5:")
display(top10)

# 6. Create grouped bar chart with two y-axes
#    Left axis = average rating
#    Right axis = total review count
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=top10["Category"],
        y=top10["Average_Rating"],
        name="Average Rating",
        marker_color="#4C78A8",
        hovertemplate=(
            "Category: %{x}<br>"
            "Average Rating: %{y:.2f}<extra></extra>"
        )
    ),
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=top10["Category"],
        y=top10["Total_Reviews"],
        name="Total Review Count",
        marker_color="#F58518",
        hovertemplate=(
            "Category: %{x}<br>"
            "Total Reviews: %{y:,.0f}<extra></extra>"
        )
    ),
    secondary_y=True
)

fig.update_layout(
    title="Task 5: Average Rating and Total Reviews by Top 10 Categories",
    xaxis_title="App Category",
    barmode="group",
    template="plotly_white",
    height=700,
    hovermode="x unified",
    legend_title="Metric",
    margin=dict(l=70, r=80, t=100, b=150)
)

fig.update_yaxes(
    title_text="Average Rating",
    range=[0, 5],
    secondary_y=False
)

fig.update_yaxes(
    title_text="Total Review Count",
    rangemode="tozero",
    secondary_y=True,
    separatethousands=True
)

fig.update_xaxes(tickangle=-35)

# Show chart in the notebook for inspection
fig.show()

# 7. Build the HTML dashboard with an IST time restriction.
#    Dashboard visible from 15:00 inclusive to 17:00 exclusive.
chart_html = fig.to_html(
    full_html=False,
    include_plotlyjs=True,
    config={"responsive": True, "displaylogo": False}
)

dashboard_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Task 5 - App Category Ratings and Reviews</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 20px;
    background: #f4f6f9;
    color: #222;
}}
.container {{
    max-width: 1400px;
    margin: auto;
    background: white;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
}}
#notice {{
    text-align: center;
    padding: 45px 20px;
    font-size: 20px;
}}
.small {{
    text-align: center;
    color: #666;
    font-size: 14px;
}}
</style>
</head>
<body>
<div class="container">
    <h2> Google Play Store Analytics</h2>
    <p class="small">
        Grouped bar chart: average rating and total reviews for the top
        10 eligible categories by installs.
    </p>
    <div id="notice" style="display:none;">
        <h3>Dashboard currently unavailable</h3>
        <p>This dashboard is available only between 3:00 PM and 5:00 PM IST.</p>
        <p id="currentTime"></p>
    </div>
    <div id="chart" style="display:none;">
        {chart_html}
    </div>
</div>

<script>
function checkDashboardTime() {{
    const parts = new Intl.DateTimeFormat("en-GB", {{
        timeZone: "Asia/Kolkata",
        hour: "2-digit",
        minute: "2-digit",
        hourCycle: "h23"
    }}).formatToParts(new Date());

    const hour = Number(parts.find(p => p.type === "hour").value);
    const minute = Number(parts.find(p => p.type === "minute").value);
    const currentMinutes = hour * 60 + minute;

    const allowed = currentMinutes >= 900 && currentMinutes < 1020;

    document.getElementById("chart").style.display =
        allowed ? "block" : "none";
    document.getElementById("notice").style.display =
        allowed ? "none" : "block";

    document.getElementById("currentTime").textContent =
        "Current time: " +
        new Intl.DateTimeFormat("en-IN", {{
            timeZone: "Asia/Kolkata",
            hour: "2-digit",
            minute: "2-digit",
            second: "2-digit",
            hour12: true
        }}).format(new Date()) + " IST";
}}

checkDashboardTime();
setInterval(checkDashboardTime, 15000);
</script>
</body>
</html>
"""

OUTPUT_FILE = Path("dashboard_task5.html")
OUTPUT_FILE.write_text(dashboard_html, encoding="utf-8")

# 8. Print summary and open the HTML dashboard
now_ist = datetime.now(ZoneInfo("Asia/Kolkata"))
print("\nTask 5 processing completed.")
print("Eligible categories:", len(top10))
print("Dashboard saved to:", OUTPUT_FILE.resolve())
print("Current IST time:", now_ist.strftime("%I:%M:%S %p"))

webbrowser.open(OUTPUT_FILE.resolve().as_uri())
print("Dashboard opened in your default browser.")
print("Display window: 3:00 PM to before 5:00 PM IST.")

Dataset loaded: (10841, 13)
Apps after size and January filters: 182

Top categories for Task 5:


,Category,Average_Rating,Total_Reviews,Total_Installs
0,FAMILY,4.185714,4737820.0,199720820.0
1,SPORTS,4.342857,1982017.0,120511000.0
2,GAME,4.251515,2412245.0,117291000.0
3,ENTERTAINMENT,4.250000,1238948.0,51000000.0
4,PERSONALIZATION,4.475000,155996.0,15060000.0
5,LIFESTYLE,4.050000,53376.0,6171500.0
6,SHOPPING,4.200000,19950.0,2000000.0
7,EDUCATION,4.400000,57645.0,2000000.0
8,COMMUNICATION,4.050000,15394.0,1010000.0
9,ART_AND_DESIGN,4.100000,2167.0,620000.0



Task 5 processing completed.
Eligible categories: 10
Dashboard saved to: C:\Users\Ranjita\Downloads\Google-Play-Store-Analytics-main (1)\Google-Play-Store-Analytics-main\dashboard_task5.html
Current IST time: 02:06:55 PM
Dashboard opened in your default browser.
Display window: 3:00 PM to before 5:00 PM IST.
